In [1]:
# %% [markdown]
# # Ray Tune 最终 Epoch 结算分析函数 (v8_s6 消融全兼容版)
# 逻辑：动态抓取 head_hidden_dims、hidden_dims、alpha 以及 v8_s6 新增的各类消融参数(renorm, ohem, clip)。

# %%
import os
import json
import pandas as pd

# %%
def analyze_ray_results(base_path):
    """
    遍历指定的 Ray 结果路径，按模型实际超参分类对比最后 Epoch 的结算指标。
    🌟 新增补齐：全面兼容 v8_s6_rnc, c, ohem, bl 消融实验参数。
    """
    print(f"===========================================================")
    print(f" 开始解析路径: {base_path}")
    print(f"===========================================================")
    
    all_trials_data = []

    # 遍历目录
    for root, dirs, files in os.walk(base_path):
        if "params.json" in files and "result.json" in files:
            params_path = os.path.join(root, "params.json")
            result_path = os.path.join(root, "result.json")
            
            # 1. 解析参数
            try:
                with open(params_path, 'r', encoding='utf-8') as f:
                    params = json.load(f)
            except Exception:
                continue
                
            # 2. 定位最后一轮的结算指标
            last_epoch_data = None
            try:
                with open(result_path, 'r', encoding='utf-8') as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if lines:
                        # 直接获取最后一行（最后 Epoch）
                        last_epoch_data = json.loads(lines[-1])
            except Exception:
                continue
                
            if last_epoch_data is not None:
                head_dims = params.get("head_hidden_dims")
                head_dims_str = str(head_dims) if head_dims is not None else "None"
                
                hidden_dims = params.get("hidden_dims")
                hidden_dims_str = str(hidden_dims) if hidden_dims is not None else "None"
                
                # 提取最终结算数据 (引入全部消融实验核心参数)
                trial_info = {
                    "trial_id": last_epoch_data.get("trial_id", os.getpid()),
                    "model": params.get("model", "None"),
                    "hidden_dims": hidden_dims_str,
                    "head_hidden_dims": head_dims_str,                    
                    "alpha_wool": params.get("conflict_alpha_wool", "None"),
                    "alpha_gold": params.get("conflict_alpha_gold", "None"),
                    "alpha_walkin": params.get("conflict_alpha_walkin", "None"),
                    "temp": params.get("ours_s6_temp", params.get("truncation_temp", params.get("align_temp", "None"))),
                    "weight_decay": params.get("weight_decay", "None"),
                    
                    # 🌟 v8_s6 核心消融参数组
                    "c_renorm": params.get("conflict_renorm", "None"),
                    "c_use_ohem": params.get("conflict_use_ohem", "None"),
                    "c_ohem_pct": params.get("conflict_ohem_pct", "None"),
                    "c_use_clip": params.get("conflict_use_weight_clip", "None"),
                    "c_max_thres": params.get("conflict_max_weight_thres", "None"),
                    "c_min_thres": params.get("conflict_min_weight_thres", "None"),
                    
                    "final_epoch": last_epoch_data.get("epoch"),
                    "final_local_best_test_Y_AUUC": last_epoch_data.get("local_best_test_Target_Y_AUUC"),
                    "final_local_best_test_C_AUUC": last_epoch_data.get("local_best_test_Target_C_AUUC"),
                    "final_loss": last_epoch_data.get("loss")
                }
                all_trials_data.append(trial_info)

    # 3. 统计并打印结果
    df = pd.DataFrame(all_trials_data)

    if df.empty:
        print("❌ 没有找到任何有效的实验数据，请检查传入的路径是否正确。\n")
        return None
        
    # 🌟 关键补齐：将新增的消融维度加入 core_params 进行严格分组
    core_params = [
        "model", "hidden_dims", "head_hidden_dims", "alpha_wool", 
        "weight_decay", "temp", 
        "c_renorm", "c_use_ohem", "c_ohem_pct", 
        "c_use_clip", "c_max_thres", "c_min_thres"
    ]
    
    for col in core_params:
        if col in df.columns:
            df[col] = df[col].fillna("None").astype(str)
        else:
            df[col] = "None"

    print(f"📊 成功加载 {len(df)} 个实验的最终结算数据。\n")
    
    # -------------------------------------------------------------
    # 维度 1：按参数组合展现各自最后 Epoch 的最高表现
    # -------------------------------------------------------------
    print("="*95)
    print(" 🎯 按照核心超参数(含消融维度)分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC ")
    print("="*95)
    
    # 过滤掉不参与对比的 None 字段以保持显示整洁
    idx = df.groupby(core_params)["final_local_best_test_Y_AUUC"].idxmax()
    df_best_per_group = df.loc[idx].sort_values(by="final_local_best_test_Y_AUUC", ascending=False)
    
    for _, row in df_best_per_group.iterrows():
        print(f"\n[配置组合] Model: {row['model']} | Hidden: {row['hidden_dims']} | Head: {row['head_hidden_dims']}")
        print(f"            基础参数 -> alpha: {row['alpha_wool']} | Temp: {row['temp']} | wd: {row['weight_decay']}")
        
        # 动态拼接消融参数字符串
        ablation_str = f"renorm={row['c_renorm']} | "
        if row['c_use_ohem'] == "True":
            ablation_str += f"OHEM=True (pct={row['c_ohem_pct']}) | "
        else:
            ablation_str += "OHEM=False | "
            
        if row['c_use_clip'] == "True":
            ablation_str += f"Clip=True (max={row['c_max_thres']}) | "
        else:
            ablation_str += "Clip=False | "
            
        if row['c_min_thres'] != "None":
            ablation_str += f"Min_Thres={row['c_min_thres']}"
            
        print(f"            消融控制 -> {ablation_str.strip(' | ')}")
        print(f"  -> 最终结算 Epoch: {row['final_epoch']} (Loss: {row['final_loss']:.4f})")
        print(f"  -> 最终 local_best_test_Y_AUUC: {row['final_local_best_test_Y_AUUC']:.6f}")
        print(f"  -> 最终 local_best_test_C_AUUC: {row['final_local_best_test_C_AUUC']:.6f}")
        print(f"  -> Trial ID: {row['trial_id']}")

    # -------------------------------------------------------------
    # 维度 2：大宽表横向对比
    # -------------------------------------------------------------
    print("\n" + "="*95)
    print(" 📋 全局最终战报总览表（按最终 local_best_test_Y_AUUC 从高到低排序） ")
    print("="*95)
    
    report_cols = core_params + ["final_epoch", "final_local_best_test_Y_AUUC", "final_local_best_test_C_AUUC"]
    
    #  pandas 显示优化
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    print(df.sort_values(by="final_local_best_test_Y_AUUC", ascending=False)[report_cols].to_string(index=False))
    print("\n")
    
    return df

In [2]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_rnc"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_rnc
📊 成功加载 18 个实验的最终结算数据。

 🎯 按照核心超参数(含消融维度)分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=none | OHEM=False | Clip=True (max=8.0)
  -> 最终结算 Epoch: 12 (Loss: 0.0514)
  -> 最终 local_best_test_Y_AUUC: 0.915181
  -> 最终 local_best_test_C_AUUC: 0.831545
  -> Trial ID: 72ea1_00016

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 10 | Temp: None | wd: 0.01
            消融控制 -> renorm=none | OHEM=False | Clip=True (max=8.0)
  -> 最终结算 Epoch: 12 (Loss: 0.0685)
  -> 最终 local_best_test_Y_AUUC: 0.914850
  -> 最终 local_best_test_C_AUUC: 0.830962
  -> Trial ID: 72ea1_00017

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=none | OHEM=False | Clip=True (max=5.0)
  -> 最终结算 Epoch: 12 (Loss: 0.

In [3]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_c"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_c
📊 成功加载 18 个实验的最终结算数据。

 🎯 按照核心超参数(含消融维度)分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=False | Clip=True (max=8.0)
  -> 最终结算 Epoch: 12 (Loss: 0.0493)
  -> 最终 local_best_test_Y_AUUC: 0.915135
  -> 最终 local_best_test_C_AUUC: 0.831784
  -> Trial ID: 72e7d_00016

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=False | Clip=True (max=5.0)
  -> 最终结算 Epoch: 12 (Loss: 0.0452)
  -> 最终 local_best_test_Y_AUUC: 0.914681
  -> 最终 local_best_test_C_AUUC: 0.832582
  -> Trial ID: 72e7d_00013

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 10 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=False | Clip=True (max=8.0)
  -> 最终结算 Epoch: 12 (Loss: 0.06

In [4]:
target_path = "/NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_ohem"

# 运行分析函数
res_df = analyze_ray_results(target_path)

 开始解析路径: /NAS/shith/uplift/ray_results/tune/criteo/train_y/TARNET/y_v8_s6_ohem
📊 成功加载 54 个实验的最终结算数据。

 🎯 按照核心超参数(含消融维度)分类：各组最终 Epoch 的最高 local_best_test_Y_AUUC 

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=True (pct=0.9) | Clip=False
  -> 最终结算 Epoch: 12 (Loss: 0.0493)
  -> 最终 local_best_test_Y_AUUC: 0.915143
  -> 最终 local_best_test_C_AUUC: 0.831709
  -> Trial ID: 72eb1_00052

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=True (pct=0.8) | Clip=False
  -> 最终结算 Epoch: 12 (Loss: 0.0493)
  -> 最终 local_best_test_Y_AUUC: 0.915123
  -> 最终 local_best_test_C_AUUC: 0.831710
  -> Trial ID: 72eb1_00046

[配置组合] Model: TARNET | Hidden: [64, 32] | Head: None
            基础参数 -> alpha: 5 | Temp: None | wd: 0.01
            消融控制 -> renorm=mean | OHEM=True (pct=0.7) | Clip=False
  -> 最终结算 Epoch: 12 (Loss: 0.